# Cluster count analysis

Since CyteType runs cost money, and each cluster costs a certain number of dollars, this notebook looks at the number of clusters present in the data

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Inspect columns
df = pd.read_csv(Path("..") / "output/clustering_pipeline/20260519_160627/run.csv")
df.columns

In [ ]:
# Total number of clusters across data
df["nClustersPostMerge"].sum()

In [ ]:
# Number of unique cluster counts
nclust = df["nClustersPostMerge"]
np.unique(nclust)

In [ ]:
# Filter dataframe, show total number of clusters after filtering
df = df[df["nClustersPostMerge"] < 100]
df = df.dropna(subset=["nClustersPostMerge"])  # pyright: ignore[reportCallIssue]
df["nClustersPostMerge"].sum()

## Disease categories per accession

Load the JSON produced by `notebooks/metadata.ipynb` (`output/metadata/accession_disease_categories.json`). Each accession maps to its raw `disease` string and the list of `DISEASE_MAP` labels matched by its disease string. An accession can match more than one label because `DISEASE_MAP` is a hierarchy.

In [ ]:
import json

categories_path = Path("..") / "output/metadata/accession_disease_categories.json"
with categories_path.open() as f:
    accession_categories = json.load(f)

categories_df = pd.DataFrame(
    [
        {"srx_accession": srx, "disease": entry["disease"], "categories": entry["categories"]}
        for srx, entry in accession_categories.items()
    ]
)
print(f"Loaded {len(categories_df):,} accessions from {categories_path}")
categories_df.head()

In [ ]:
# Frequency of each label across accessions (counts every match, so an accession
# that hit Lung Cancer + NSCLC + LUAD contributes to all three).
label_counts = (
    categories_df.explode("categories")
    .dropna(subset=["categories"])
    .groupby("categories")
    .size()
    .sort_values(ascending=False)
)
label_counts

In [ ]:
# Join categories onto the clustering run frame and show how many run.csv accessions
# have a category mapping at all (the JSON covers only the lung intersection).
clusters_with_categories = df.merge(
    categories_df, left_on="srx", right_on="srx_accession", how="left"
)
n_mapped = clusters_with_categories["categories"].notna().sum()
print(f"{n_mapped:,} of {len(clusters_with_categories):,} run.csv rows have a disease-category entry")
clusters_with_categories[["srx", "nClustersPostMerge", "disease", "categories"]].head()

In [ ]:
# Sum of nClustersPostMerge per disease label. An accession with multiple labels
# (e.g. Lung Cancer + NSCLC + LUAD) contributes its cluster count to each of them.
clusters_per_label = (  # pyright: ignore[reportCallIssue]
    clusters_with_categories.dropna(subset=["categories", "nClustersPostMerge"])
    .explode("categories")
    .groupby("categories")["nClustersPostMerge"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)
clusters_per_label

### Disjoint partition (most-specific label wins)

Each accession is assigned to a single bucket: the most-specific label in its `categories` list. Because `DISEASE_MAP` is ordered parents-before-children, the most-specific label is the last entry. Accessions with no category go to `Other`. Same logic as `plot_disease_breakdown`.

In this view every accession contributes its cluster count exactly once, so columns are non-overlapping and add up to a true total.

In [ ]:
_ready = clusters_with_categories.dropna(subset=["categories", "nClustersPostMerge"]).copy()
_ready["mostSpecificLabel"] = _ready["categories"].map(
    lambda cats: cats[-1] if isinstance(cats, list) and len(cats) > 0 else "Other"
)

clusters_per_label_disjoint = (
    _ready.groupby("mostSpecificLabel")["nClustersPostMerge"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)
clusters_per_label_disjoint

In [ ]:
label_comparison = (
    pd.concat(
        [clusters_per_label.rename("inclusive"), clusters_per_label_disjoint.rename("disjoint")],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
label_comparison["overlapAbsorbed"] = label_comparison["inclusive"] - label_comparison["disjoint"]
label_comparison.sort_values("inclusive", ascending=False)

## Cluster counts in the disease-balanced subset

Load `accession_disease_categories_subset.json` (built in `notebooks/metadata.ipynb`), intersect with `run.csv`, and inspect cluster counts per label. The subset is already disjoint by construction (each accession has exactly one most-specific label drawn from IPF / COVID / COPD / ILD / CF), so per-label sums add up cleanly to a single total.

In [ ]:
subset_path = Path("..") / "output/metadata/accession_disease_categories_subset.json"
with subset_path.open() as f:
    subset_categories = json.load(f)

subset_df = pd.DataFrame(
    [
        {
            "srx_accession": srx,
            "disease": entry["disease"],
            "mostSpecificLabel": entry["categories"][-1] if entry["categories"] else "Other",
        }
        for srx, entry in subset_categories.items()
    ]
)
clusters_subset = df.merge(subset_df, left_on="srx", right_on="srx_accession", how="inner")
print(f"{len(clusters_subset):,} of {len(subset_df):,} subset accessions present in run.csv")
clusters_subset.head()

In [ ]:
subset_clusters_per_label = (
    clusters_subset.groupby("mostSpecificLabel")["nClustersPostMerge"]
    .agg(nAccessions="count", nClusters="sum")
    .astype(int)
    .sort_values("nClusters", ascending=False)
)
total_clusters = int(clusters_subset["nClustersPostMerge"].sum())
total_accessions = len(clusters_subset)
print(f"total: {total_clusters:,} clusters across {total_accessions:,} accessions")
subset_clusters_per_label